In [9]:
import pandas as pd
import os
import zipfile
import re

In [2]:
data = pd.read_csv("data/metadonnees.csv")
data.head()

C:\Users\auran\AppData\Local\Temp\ipykernel_19076\3855839679.py:1: DtypeWarning: Columns (8,9,10,12,28,29,30,31,32,33,34,35,36,37,38,39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("data/metadonnees.csv")


,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations
0,EL009_L_1958_11_001_01_1_PF_01,1958-11-23,France;Élections législatives;Assemblée Nation...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,non mentionné,non mentionné,cultivateur,maire;conseiller général,non mentionné,non mentionné,non mentionné,Parti radical,non mentionné,non
1,EL009_L_1958_11_001_01_1_PF_02,1958-11-23,France;Ve République;Élections législatives;As...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,non mentionné,non mentionné,cultivateur,conseiller municipal,non mentionné,non mentionné,prisonnier de guerre,Union pour la nouvelle République,non mentionné,non
2,EL009_L_1958_11_001_01_1_PF_03,1958-11-23,Élections législatives;France;Assemblée Nation...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,non mentionné,non mentionné,cultivateur,non mentionné,non mentionné,non mentionné,non mentionné,Parti communiste français,non mentionné,non
3,EL009_L_1958_11_001_01_1_PF_04,1958-11-23,Élections législatives;France;Assemblée Nation...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,35,entre 30 et 39 ans,greffier de paix,conseiller municipal;conseiller général,non mentionné,non mentionné,combattant,non mentionné,non mentionné,oui
4,EL009_L_1958_11_001_01_1_PF_05,1958-11-23,Ve République;Assemblée Nationale;Élections lé...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,non mentionné,non mentionné,cultivateur;président Coopérative élevage,non mentionné,non mentionné,non mentionné,non mentionné,Centre national des indépendants et paysans,non mentionné,non


In [3]:
data.columns

Index(['id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour',
       'cote', 'departement', 'departement-nom', 'departement-insee',
       'identifiant de circonscription', 'images', 'pdf', 'ocr_url',
       'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age',
       'titulaire-age-calcule', 'titulaire-age-tranche',
       'titulaire-profession', 'titulaire-mandat-en-cours',
       'titulaire-mandat-passe', 'titulaire-associations',
       'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste',
       'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom',
       'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule',
       'suppleant-age-tranche', 'suppleant-profession',
       'suppleant-mandat-en-cours', 'suppleant-mandat-passe',
       'suppleant-associations', 'suppleant-autres-statuts',
       'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations'],
      dtype='object')

In [4]:
data["date"] = pd.to_datetime(data["date"])

data[["annee","mois","jour"]] = data["date"].apply(
    lambda x: pd.Series([x.year, x.month, x.day])
)
print("Nombre de professions par années :", data["annee"].value_counts().sort_index())

Nombre de professions par années : annee
1958    2774
1959       7
1962    2235
1965       8
1967    2871
1968    2872
1969       8
1973    3843
1974      13
1978    4830
1979       7
1981    3133
1986     740
1988    3551
1989      13
1993    5837
1995      11
1999      14
2002      18
2004      50
2007      13
2009      60
2012      12
2014      97
2019      13
Name: count, dtype: int64


In [6]:
# Export des professions de foi
zip_path = "C:/Users/auran/OneDrive/Documents/ensae/3A/NLP/legislatives.zip"

docs = []

with zipfile.ZipFile(zip_path) as z:
    for file in z.namelist():
        if file.endswith(".txt"):
            with z.open(file) as f:
                text = f.read().decode("utf-8", errors="ignore")
                docs.append({"file": file, "text": text})

transcriptions = pd.DataFrame(docs)

transcriptions = transcriptions.assign(
    id=transcriptions["file"].str.extract(r'([^/]+)\.txt'),
    annee=transcriptions["file"].str.extract(r'text_files/(\d{4})')
)
transcriptions = transcriptions.drop(columns = ["file"])

In [7]:
transcriptions.head()

,text,id,annee
0,Département de Seine-Maritime - 12ème Circonsc...,EL196_L_1993_03_076_12_1_PF_01,1993
1,ELECTIONS LEGISLATIVES DU 21 MARS 1993\nREPUBL...,EL190_L_1993_03_024_02_1_PF_03,1993
2,Sciences Po / fonds CEVIPOF\nELECTIONS LEGISLA...,EL190_L_1993_03_017_01_1_PF_07,1993
3,Sciences Po / fonds CEVIPOF\nENTENTE DES ECOLO...,EL192_L_1993_03_050_02_1_PF_03,1993
4,Sciences Po / fonds CEVIPOF\nLes Verts Confédé...,EL196_L_1993_03_079_04_1_PF_03,1993


In [11]:
pd.set_option('display.max_colwidth', None)
extrait = transcriptions.sample()

In [26]:
def nettoyage_profession_foi(texte):

    if not isinstance(texte, str):
        return ""
    
    texte = re.sub(r'[\u2600-\u26FF\u2700-\u27BF\u25A0-\u25FF]', ' ', texte)
    texte = re.sub(r'[A-Z]{4,}\s[A-Z]{4,}', '', texte) 

    # Enlever les 'ELECTIONS LEGISLATIVES'
    texte = re.sub(r'ÉLECTIONS\s[A-Z\s]+-\s[A-Z]+\s\d{4}', '', texte, flags=re.IGNORECASE)

    # Suppression des noms de candidats/suppléants
    texte = re.sub(r'\n?[A-Z][a-z]+\s+[A-Z]{2,}\s+(?:candidat|suppléant)[^\n]*', '', texte) # Suppression des noms de candidats/suppléants
    
    # Suppression de la mention 'Sciences Po/ fonds CEVIPOF'
    texte = re.sub(r'.*?(?:Sciences Po|fonds CEVIPOF|Archives).*?$', '', texte, flags=re.MULTILINE | re.IGNORECASE)
    
    # Nettoyage des espaces et retours à la ligne
    texte = re.sub(r'\n{3,}', '\n\n', texte) 
    texte = re.sub(r'\s+', ' ', texte) 
    
    # Suppression des césures
    texte = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', texte)

    texte = texte.lower()
    
    return texte.strip()



In [23]:
# Test 
print(nettoyage_profession_foi(extrait['text'].iloc[0]))


élections législatives - 21 mars 1993 - 16ème circonscription georges hage député, vice-président de l'assemblée nationale aldebert valette conseiller général maire d'auby député suppléant c'est mieux ! que ça change. la jeunesse est bafouée dans ses droits et ses espérances. le chômage s'étend. vous connaissez des fins de mois difficiles, l'insécurité et vous craignez pour l'avenir de vos enfants. la . cette politique qui a échoué, ce n'est pas une politique de gauche, mais une politique toute au service du grand patronat, de la bourse et des spéculateurs, que la droite s'apprête à aggraver. battue en 1981, de nouveau battue en 1988, la voici, cette droite, qui crie victoire et annonce de concert avec le grand patronat la poursuite des privatisations, encore plus de rigueur et d'austérité, de bas salaires et de chômage. les communistes ont toujours été les adversaires politiques les plus résolus de la droite et de sa politique. pour la combattre, pour mettre en chantier avec vous une 

In [27]:
transcriptions['texte_nettoye'] = transcriptions['text'].apply(nettoyage_profession_foi)